# Tuning de las tres primeras familias por negocio

Preparado sin salidas hasta que termine `tuning`. Los espacios y las semillas fueron fijados antes de la comparación definitiva. Se usa RandomizedSearchCV con configuraciones preextraídas: controla el presupuesto y permite escalas logarítmicas sin una grilla desproporcionada. Cada configuración es una corrida hija de su familia, con cinco folds y refit.

La selección del top 3 usa cobertura de Fail, no AP. Si el LLM ocupa una de esas posiciones, participa con sus treinta configuraciones; su tuning ajusta ejemplos y temperatura local de puntuación, no los pesos del GPT.


In [ ]:
from pathlib import Path
import os, sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents] if (p/'config/protocolo-v2.json').is_file()), None)
if ROOT is None and (here/'pset-03/v2/config/protocolo-v2.json').is_file():
    ROOT = here/'pset-03/v2'
assert ROOT is not None, 'Abrir el notebook desde v2/notebooks o desde la raíz del repositorio'
os.environ['PSET3_V2_ROOT'] = str(ROOT)
sys.path.insert(0, str(ROOT))
from src.seguimiento import protocolo
from src.common import sha256
plan = protocolo()
def report(relative):
    path = ROOT/relative
    if not path.is_file():
        raise RuntimeError('Etapa pendiente: falta '+relative+'. Revisar docs/estado-v2.json y esperar al coordinador; no sustituir por resultados de v1.')
    return json.loads(path.read_text())
print('Protocolo verificado:', plan['id'])
print('Este notebook consulta resultados reales y reproduce sus cálculos; no inicia nuevas búsquedas ni llamadas de inferencia API.')


In [ ]:
tuned = report('docs/tuning/resultados.json')
search = report('docs/tuning/plan.json')
assert tuned['status']=='complete' and tuned['protocol_id']==plan['id']
assert len(tuned['families'])==3 and len(tuned['configurations'])>=90
assert tuned['holdout_evaluated'] is False
from src.tuning import choose
from src.negocio import normalized
models = report('docs/modelos/resultados.json')
summary=[]
for family in tuned['families']:
    trials=[normalized(r) for r in tuned['configurations'] if r['family']==family]
    assert len(trials)>=30 and len({json.dumps(r['params'],sort_keys=True) for r in trials})>=30
    default=next(r for r in models['candidates'] if r['family']==family)
    best=choose(trials)
    spread=max(default['cv_coverage_std'],best['cv_coverage_std'])
    delta=best['cv_coverage_mean']-default['cv_coverage_mean']
    summary.append({'familia':family,'configuraciones':len(trials),'cobertura_default':default['cv_coverage_mean'],
                    'cobertura_tuned':best['cv_coverage_mean'],'DE_tuned':best['cv_coverage_std'],
                    'delta':delta,'variacion_referencia':spread,'mejora_supera_variacion':delta>spread,
                    'AP_tuned':best['cv_ap_fail_mean'],'run_id':best['run_id'],'parametros':best['params']})
    print(family,'| valores predeclarados de cada parámetro:')
    draws=search['content']['draws'][family]
    display(pd.DataFrame(draws).describe(include='all').T)
display(pd.DataFrame(summary))


In [ ]:
trials_table=pd.DataFrame([{k:v for k,v in r.items() if k not in ['folds','warnings']} for r in tuned['configurations']])
display(trials_table[['family','config_index','cv_coverage_mean','cv_coverage_std','cv_ap_fail_mean','train_ap_fail_mean','ap_gap','search_seconds','run_id']].sort_values(['family','cv_coverage_mean'],ascending=[True,False]))


## Decisión sobre la mejora

Se informa el delta frente al default y se contrasta con la mayor desviación de ambos modelos. Ese criterio es descriptivo, tal como pide la rúbrica; no constituye una prueba de significancia con folds independientes. La elección de la mayor media de cobertura se distingue de la afirmación de una mejora estable. Los defaults siguen siendo elegibles si el tuning no los supera. Ningún resultado de prueba se utiliza para cambiar esta decisión.
